The model learns to predict both YEAR_n and event tokens.

Everything is in a single flat vocabulary.

Loss is standard CrossEntropyLoss.

Generation works out-of-the-box: you can autoregressively generate tokens until "Died".



In [2]:
import pandas as pd
import numpy as np

In [23]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.utils.data import Dataset


1. Read the data and define vocabularies

In [17]:
# read training data, create main vocab from all events in the data, create year_vocab from 0 to 110
df = pd.read_csv("Event_data.csv")

In [18]:
event_tokens = [c for c in set(df.event)]
year_tokens = [f'YEAR_{i}' for i in range(100)]
all_tokens = event_tokens + year_tokens

main_vocab = {token: idx for idx, token in enumerate(all_tokens)}
id_to_token = {idx: token for token, idx in main_vocab.items()}

2. Updated Dataset


In [24]:
class LifeEventDatasetFlat(Dataset):
    def __init__(self, df, vocab):
        self.vocab = vocab
        self.sequences = []
        assert {'patid', 'age', 'event'}.issubset(df.columns)

        grouped = df.groupby('patid')
        for _, group in grouped:
            group = group.sort_values('age')
            tokens = []
            for _, row in group.iterrows():
                year_token = f"YEAR_{int(row['age'])}"
                event_token = row['event']
                if year_token in vocab and event_token in vocab:
                    tokens.append(year_token)
                    tokens.append(event_token)
                else:
                    raise ValueError(f"Unknown token in vocab: {year_token} or {event_token}")
            token_ids = [vocab[tok] for tok in tokens]
            self.sequences.append(torch.tensor(token_ids, dtype=torch.long))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


3. Padding Collate Function

In [27]:
def collate_flat_batch(batch, pad_token_id=0):
    max_len = max(seq.size(0) for seq in batch)
    padded_inputs = torch.full((len(batch), max_len), pad_token_id, dtype=torch.long)

    for i, seq in enumerate(batch):
        padded_inputs[i, :seq.size(0)] = seq

    return padded_inputs


4. Simple Transformer Model

In [29]:
class SimpleTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, max_seq_len=256):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)

        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead)
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, L = x.size()
        positions = torch.arange(L, device=x.device).unsqueeze(0).expand(B, L)
        x = self.embedding(x) + self.pos_embedding(positions)

        # transformer expects (L, B, D)
        x = x.transpose(0, 1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(L).to(x.device)
        x = self.transformer(x, x, tgt_mask=tgt_mask)
        x = x.transpose(0, 1)  # back to (B, L, D)

        return self.output_head(x)  # (B, L, vocab_size)


5. Training Setup

In [137]:
dataset = LifeEventDatasetFlat(df, main_vocab)
loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=collate_flat_batch)


In [140]:
model = SimpleTransformerLM(vocab_size=len(main_vocab))
criterion = nn.CrossEntropyLoss(ignore_index=main_vocab['YEAR_0'])  # or pad token ID
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    for batch in loader:
        inputs = batch[:, :-1]
        targets = batch[:, 1:]

        logits = model(inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch%10 ==9:
        print(f"Epoch {epoch + 1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 0.1344
Epoch 20, Loss: 0.0212
Epoch 30, Loss: 0.0115
Epoch 40, Loss: 0.0077
Epoch 50, Loss: 0.0083


In [141]:
targets[0]

tensor([ 4, 29,  2, 32,  3])

6. Generating samples

In [142]:
def generate_n_sequences(model, start_tokens, vocab, id_to_token, max_length=100, stop_token="died", device='cpu', k=5, n_seq = 10):
    for i in range(n_seq):
        generated_tokens = generate_sequence(model, start_tokens, vocab, id_to_token, max_length, stop_token, device, k)
        print("Generated:", " ".join(generated_tokens))
    return None


def generate_sequence(model, start_tokens, vocab, id_to_token, max_length=100, stop_token="died", device='cpu', k=5):
    model.eval()
    input_ids = [vocab[tok] for tok in start_tokens]
    input_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)  # (1, L)

    generated = input_ids.copy()

    for _ in range(max_length):
        with torch.no_grad():
            logits = model(input_tensor)  # (1, L, vocab_size)
            next_token_logits = logits[0, -1]  # (vocab_size,)

            # Top-k sampling
            topk = torch.topk(next_token_logits, k=k)
            # print(topk)
            # probs = torch.nn.functional.softmax(topk.values, dim=-1)
            # next_token_id = topk.indices[torch.multinomial(probs, num_samples=1).item()].item()
            next_token_id = topk.indices[torch.randint(0, k, (1,)).item()].item() # we choose randomly with equal probs from top k
            next_token = id_to_token[next_token_id]
            # print(next_token_id, next_token)

        generated.append(next_token_id)
        if next_token == stop_token:
            break

        input_tensor = torch.tensor(generated, dtype=torch.long).unsqueeze(0).to(device)

    return [id_to_token[tok_id] for tok_id in generated]


In [144]:
df.shape

(38, 3)

In [145]:
logits[0].shape

torch.Size([5, 105])

m = nn.Softmax(dim=1)
input = torch.randn(2, 3)
output = m(logits[0])
output[0]

In [147]:
prompt = ["YEAR_0", "born"]
generated_tokens = generate_sequence(model, prompt, main_vocab, id_to_token, max_length =20, k = 2)
print("Generated:", " ".join(generated_tokens))


Generated: YEAR_0 born YEAR_15 died


In [148]:
generate_n_sequences(model, prompt, main_vocab, id_to_token, max_length =20, k = 3, n_seq = 100)

Generated: YEAR_0 born YEAR_24 died
Generated: YEAR_0 born YEAR_80 flu YEAR_82 cancer YEAR_45 died
Generated: YEAR_0 born YEAR_80 flu YEAR_45 died
Generated: YEAR_0 born YEAR_15 died
Generated: YEAR_0 born YEAR_15 cancer YEAR_48 cancer YEAR_47 died
Generated: YEAR_0 born YEAR_24 YEAR_15 YEAR_45 died
Generated: YEAR_0 born YEAR_15 cancer YEAR_48 cancer cancer died
Generated: YEAR_0 born YEAR_80 YEAR_27 YEAR_45 flu YEAR_80 died
Generated: YEAR_0 born YEAR_24 YEAR_15 YEAR_27 flu YEAR_63 YEAR_27 YEAR_48 flu YEAR_24 YEAR_63 flu YEAR_27 died
Generated: YEAR_0 born YEAR_80 YEAR_27 YEAR_45 cancer YEAR_47 YEAR_45 YEAR_90 cancer cancer YEAR_45 flu YEAR_63 died
Generated: YEAR_0 born YEAR_15 cancer cancer YEAR_45 flu YEAR_55 flu flu died
Generated: YEAR_0 born YEAR_80 flu YEAR_82 cancer YEAR_47 cancer YEAR_55 died
Generated: YEAR_0 born YEAR_15 flu YEAR_82 flu YEAR_82 died
Generated: YEAR_0 born YEAR_15 flu flu YEAR_63 YEAR_74 died
Generated: YEAR_0 born YEAR_80 flu YEAR_27 flu YEAR_27 flu YEAR_4